In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from datetime import datetime

from susse.api_clients import ModisDataFetcher, ModisProductFactory, ModisProductEnum
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="SuSSe")
location = geolocator.geocode("Kampala")
start_date = datetime(2010, 1, 1)
end_date = datetime(2010, 1, 30)

data_fetcher = ModisDataFetcher()

factory = ModisProductFactory()
product = factory.get_product_by_enum(ModisProductEnum.LAND_SURFACE_TEMPERATURE)
result = data_fetcher.fetch_temp_day(
    location, start_date, end_date)


KeyboardInterrupt



In [2]:
from susse.api_clients import Merra2Config, MerraProducts, MerraDownloadManager
from geopy.geocoders import Nominatim
from datetime import datetime

geolocator = Nominatim(user_agent="SuSSe")
location = geolocator.geocode("Kampala")
start_date = datetime(2010, 1, 1)
download_url = Merra2Config.generate_download_link(start_date, MerraProducts.HUR.value, location.latitude, location.longitude)

download_manager = MerraDownloadManager()
download_manager.download_from_urls(download_url)

ValueError: Authorizing session failed due to HTTP error.HTTP Error 404: 404

In [6]:
from susse.api_clients import Merra2Config, MerraProducts, MerraDownloadManager, MerraDataFetcher
from geopy.geocoders import Nominatim
from datetime import datetime
geolocator = Nominatim(user_agent="SuSSe")
location = geolocator.geocode("Kampala")
start_date = datetime(2010, 1, 1)
end_date = datetime(2010, 1, 3)
data_fetcher = MerraDataFetcher()
data = data_fetcher.fetch_product_result(MerraProducts.HUR.value, location, start_date, end_date)
print("managed")

ValueError: Authorizing session failed due to HTTP error.HTTP Error 503: Service Unavailable

In [2]:
from susse.api_clients import Merra2Config, MerraProducts
from geopy.geocoders import Nominatim
from datetime import datetime
geolocator = Nominatim(user_agent="SuSSe")
location = geolocator.geocode("Kampala")
start_date = datetime(2010, 1, 1)
url = Merra2Config.generate_download_link(start_date, MerraProducts.HUSS.value, location.latitude, location.longitude)
print(url)

https://goldsmr4.gesdisc.eosdis.nasa.gov/opendap/MERRA2/M2T1NXSLV.5.12.4/2010/01/MERRA2_300.tavg1_2d_slv_Nx.20100101.nc4.nc4?QV2M[0:1:23][181:1:181][245:1:245]


In [4]:
__author__ = "Jan Urbansky"

# TODO: Change and describe structure of the links that have to be provided.
# TODO: Proper readme with examples.

from multiprocessing.dummy import Pool as Threadpool
import requests
import logging
import yaml
import os
import urllib.response
from http import cookiejar
import urllib.error
import urllib.request
import re

log = logging.getLogger('opendap_download')




class DownloadManager(object):
    __AUTHENTICATION_URL = 'https://urs.earthdata.nasa.gov/oauth/authorize'
    __username = ''
    __password = ''
    __download_urls = []
    __download_path = ''
    _authenticated_session = None

    def __init__(self, username='', password='', links=None, download_path='download'):
        self.set_username_and_password(username, password)
        self.download_urls = links
        self.download_path = download_path

        if logging.getLogger().getEffectiveLevel() == logging.INFO:
            logging.getLogger("requests").setLevel(logging.CRITICAL)
            logging.getLogger("urllib3").setLevel(logging.CRITICAL)
        
        log.debug('Init DownloadManager')

    @property
    def download_urls(self):
        return self.__download_urls

    @download_urls.setter
    def download_urls(self, links):
        """
        Setter for the links to download. The links have to be an array containing the URLs. The module will
        figure out the filename from the url and save it to the folder provided with download_path()
        :param links: The links to download
        :type links: List[str]
        """
        # TODO: Check if links have the right structure? Read filename from links?
        # Check if all links are formed properly
        if links is None:
            self.__download_urls = []
        else:
            for item in links:
                try:
                    self.get_filename(item)
                except AttributeError:
                    raise ValueError('The URL seems to not have the right structure: ', item)
            self.__download_urls = links

    @property
    def download_path(self):
        return self.__download_path

    @download_path.setter
    def download_path(self, file_path):
        self.__download_path = file_path

    def set_username_and_password(self, username, password):
        self.__username = username
        self.__password = password

    def read_credentials_from_yaml(self, file_path_to_yaml):
        with open(file_path_to_yaml, 'r') as f:
            credentials = yaml.load(f)
            log.debug('Credentials: ' + str(credentials))
            self.set_username_and_password(credentials['username'], credentials['password'])

    def _mp_download_wrapper(self, url_item):
        """
        Wrapper for parallel download. The function name cannot start with __ due to visibility issues.
        :param url_item:
        :type url_item:
        :return:
        :rtype:
        """
        query = url_item
        file_path = os.path.join(self.download_path, self.get_filename(query))
        self.__download_and_save_file(query, file_path)

    def start_download(self, nr_of_threads=4):
        if self._authenticated_session is None:
            self._authenticated_session = self.__create_authenticated_sesseion()
        # Create the download folder.
        os.makedirs(self.download_path, exist_ok=True)
        # p = multiprocessing.Pool(nr_of_processes)
        p = Threadpool(nr_of_threads)
        p.map(self._mp_download_wrapper, self.download_urls)
        p.close()
        p.join()

    @staticmethod
    def get_filename(url):
        """
        Extracts the filename from the url. This method can also be used to check
        if the links have the correct structure
        :param url: The MERRA URL
        :type url: str
        :return: The filename
        :rtype: str
        """
        # Extract everything between a leading / and .nc4? . The problem with using this without any
        # other classification is, that the URLs have multiple / in their structure. The expressions [^/]* matches
        # everything but /. Combined with the outer expressions, this only matches the part between the last / and .nc4?
        reg_exp = r'(?<=/)[^/]*(?=.nc4?)'
        file_name = re.search(reg_exp, url).group(0)
        return file_name

    def __download_and_save_file(self, url, file_path):
        r = self._authenticated_session.get(url, stream=True)
        with open(file_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1024):
                if chunk:
                    f.write(chunk)
        return r.status_code

    def __create_authenticated_sesseion(self):
        s = requests.Session()
        s.headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 6.1; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/40.0.2214.85 Safari/537.36'}
        s.auth = (self.__username, self.__password)
        s.cookies = self.__authorize_cookies_with_urllib()

        if logging.getLogger().getEffectiveLevel() == logging.DEBUG:
            r = s.get(self.download_urls[0])
            log.debug('Authentication Status')
            log.debug(r.status_code)
            log.debug(r.headers)
            log.debug(r.cookies)

            log.debug('Sessions Data')
            log.debug(s.cookies)
            log.debug(s.headers)
        return s

    def __authorize_cookies_with_urllib(self):
        username = self.__username
        password = self.__password
        top_level_url = "https://urs.earthdata.nasa.gov"

        # create an authorization handler
        p = urllib.request.HTTPPasswordMgrWithDefaultRealm()
        p.add_password(None, top_level_url, username, password);

        auth_handler = urllib.request.HTTPBasicAuthHandler(p)
        auth_cookie_jar = cookiejar.CookieJar()
        cookie_jar = urllib.request.HTTPCookieProcessor(auth_cookie_jar)
        opener = urllib.request.build_opener(auth_handler, cookie_jar)

        urllib.request.install_opener(opener)

        try:
            # The merra portal moved the authentication to the download level. Before this change you had to
            # provide username and password on the overview page. For example:
            # goldsmr4.sci.gsfc.nasa.gov/opendap/MERRA2/M2T1NXSLV.5.12.4/
            # authentication_url = 'https://goldsmr4.sci.gsfc.nasa.gov/opendap/MERRA2/M2T1NXSLV.5.12.4/1980/01/MERRA2_100.tavg1_2d_slv_Nx.19800101.nc4.ascii?U2M[0:1:1][0:1:1][0:1:1]'
            # Changes:
            # Authenticate with the first url in the links.
            # Request the website and initialiaze the BasicAuth. This will populate the auth_cookie_jar
            authentication_url = self.download_urls[0]
            result = opener.open(authentication_url)
            log.debug(list(auth_cookie_jar))
            log.debug(list(auth_cookie_jar)[0])
            log.debug(list(auth_cookie_jar)[1])

        except urllib.error.HTTPError:
            raise ValueError('Username and or Password are not correct!')
        except IOError as e:
            log.warning(e)
            raise IOError
        except IndexError as e:
            log.warning(e)
            raise IndexError('download_urls is not set')

        return auth_cookie_jar

# 
# if __name__ == '__main__':
#     link = [
#         'http://goldsmr4.sci.gsfc.nasa.gov:80/opendap/MERRA2/M2T1NXSLV.5.12.4/2014/01/MERRA2_400.tavg1_2d_slv_Nx.20140101.nc4.nc4?U2M[0:1:5][358:1:360][573:1:575],U10M[0:1:5][358:1:360][573:1:575],U50M[0:1:5][358:1:360][573:1:575],V2M[0:1:5][358:1:360][573:1:575],V10M[0:1:5][358:1:360][573:1:575],V50M[0:1:5][358:1:360][573:1:575]']
# 
#     logging.basicConfig(level=logging.DEBUG, handlers=[logging.StreamHandler()])
#     dl = DownloadManager()
#     dl.download_path = 'downlaod123'
#     dl.read_credentials_from_yaml((os.path.join(os.path.dirname(os.path.realpath(__file__)), 'authentication.yaml')))
#     dl.download_urls = link
#     dl.start_download()

In [5]:
# Imports
import os
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from calendar import monthrange

####### INPUTS - CHANGE THESE #########
username = 'mijan' # Username for MERRA download account
password = 'MarconiLafM2024!' # Password for MERRA download account
years = [2007, 2008, 2009, 2010, 2011] # List of years for which data will be downloaded
years = [2010]
field_id = 'T2M' # ID of field in MERRA-2 - find ID here: https://gmao.gsfc.nasa.gov/pubs/docs/Bosilovich785.pdf
field_id = 'QV2M' # ID of field in MERRA-2 - find ID here: https://gmao.gsfc.nasa.gov/pubs/docs/Bosilovich785.pdf 
field_name = 'temperature' # Name of field to be stored with downloaded data (can use any name you like)
field_name = 'tmp' # Name of field to be stored with downloaded data (can use any name you like)
database_name = 'M2I1NXASM' # Name of database in which field is stored, can be looked up by ID here: https://gmao.gsfc.nasa.gov/pubs/docs/Bosilovich785.pdf 
database_name  = 'M2T1NXSLV'
database_id = 'inst1_2d_asm_Nx' # ID of database database in which field is stored, also can be looked up by ID here: https://gmao.gsfc.nasa.gov/pubs/docs/Bosilovich785.pdf 
database_id = 'tavg1_2d_slv_Nx' # ID of database database in which field is stored, also can be looked up by ID here: https://gmao.gsfc.nasa.gov/pubs/docs/Bosilovich785.pdf 
locs = [('maputo', -25.9629, 32.5732), # List of locations for which data will be downloaded. Each location is a three-tuple, consisting of name (string), latitude, and longitude floats)
        ('cdelgado', -12.3335, 39.3206), 
        ('manica', -18.9438, 32.8649),
        ('gaza', -23.0222, 32.7181),
        ('sofala', -19.2039, 34.8624),
        ('tete', -16.1328, 33.6364),
        ('zambezia', -16.5639, 36.6094),
        ('nampula', -15.1266, 39.2687),
        ('niassa', -12.7826, 36.6094),
        ('inhambane', -23.8662, 35.3827)]
locs = [('kampala', location.latitude, location.longitude)]
conversion_function = lambda x: x - 273.15 # Unit conversion function to be applied to daily data. Here is the unit conversion for temperature from Kelvin to Celsius. 
aggregator = 'mean' # Method by which data will be aggregated over days and weeks. Can be "sum", "mean", "min", or "max" (for example, mean will download hourly data, mean daily data, and mean weekly data)

####### CONSTANTS - DO NOT CHANGE BELOW THIS LINE #######
lat_coords = np.arange(0, 361, dtype=int)
lon_coords = np.arange(0, 576, dtype=int)
database_url = 'https://goldsmr4.gesdisc.eosdis.nasa.gov/opendap/MERRA2/' + database_name + '.5.12.4/'
NUMBER_OF_CONNECTIONS = 5

####### DOWNLOAD DATA #########
# Translate lat/lon into coordinates that MERRA-2 understands
def translate_lat_to_geos5_native(latitude):
    """
    The source for this formula is in the MERRA2 
    Variable Details - File specifications for GEOS pdf file.
    The Grid in the documentation has points from 1 to 361 and 1 to 576.
    The MERRA-2 Portal uses 0 to 360 and 0 to 575.
    latitude: float Needs +/- instead of N/S
    """
    return ((latitude + 90) / 0.5)

def translate_lon_to_geos5_native(longitude):
    """See function above"""
    return ((longitude + 180) / 0.625)

def find_closest_coordinate(calc_coord, coord_array):
    """
    Since the resolution of the grid is 0.5 x 0.625, the 'real world'
    coordinates will not be matched 100% correctly. This function matches 
    the coordinates as close as possible. 
    """
    # np.argmin() finds the smallest value in an array and returns its
    # index. np.abs() returns the absolute value of each item of an array.
    # To summarize, the function finds the difference closest to 0 and returns 
    # its index. 
    index = np.abs(coord_array-calc_coord).argmin()
    return coord_array[index]

def translate_year_to_file_number(year):
    """
    The file names consist of a number and a meta data string. 
    The number changes over the years. 1980 until 1991 it is 100, 
    1992 until 2000 it is 200, 2001 until 2010 it is  300 
    and from 2011 until now it is 400.
    """
    file_number = ''
    
    if year >= 1980 and year < 1992:
        file_number = '100'
    elif year >= 1992 and year < 2001:
        file_number = '200'
    elif year >= 2001 and year < 2011:
        file_number = '300'
    elif year >= 2011:
        file_number = '400'
    else:
        raise Exception('The specified year is out of range.')
    return file_number

def generate_url_params(parameter, time_para, lat_para, lon_para):
    """Creates a string containing all the parameters in query form"""
    parameter = map(lambda x: x + time_para, parameter)
    parameter = map(lambda x: x + lat_para, parameter)
    parameter = map(lambda x: x + lon_para, parameter)
    return ','.join(parameter)
    
def generate_download_links(download_years, base_url, dataset_name, url_params):
    """
    Generates the links for the download. 
    download_years: The years you want to download as array. 
    dataset_name: The name of the data set. For example tavg1_2d_slv_Nx
    """
    urls = []
    for y in download_years: 
        y_str = str(y)
        file_num = translate_year_to_file_number(y)
        for m in range(1,13):
            m_str = str(m).zfill(2)
            _, nr_of_days = monthrange(y, m)
            for d in range(1,nr_of_days+1):
                d_str = str(d).zfill(2)
                # Create the file name string
                file_name = 'MERRA2_{num}.{name}.{y}{m}{d}.nc4'.format(
                    num=file_num, name=dataset_name, 
                    y=y_str, m=m_str, d=d_str)
                # Create the query
                query = '{base}{y}/{m}/{name}.nc4?{params}'.format(
                    base=base_url, y=y_str, m=m_str, 
                    name=file_name, params=url_params)
                urls.append(query)
    return urls

print('DOWNLOADING DATA FROM MERRA')
print('Predicted time: ' + str(len(years)*len(locs)*6) + ' minutes')
print('=====================')
for loc, lat, lon in locs:
    print('Downloading ' + field_name + ' data for ' + loc)
    # Translate the coordinates that define your area to grid coordinates.
    lat_coord = translate_lat_to_geos5_native(lat)
    lon_coord = translate_lon_to_geos5_native(lon)
    # Find the closest coordinate in the grid.
    lat_closest = find_closest_coordinate(lat_coord, lat_coords)
    lon_closest = find_closest_coordinate(lon_coord, lon_coords)
    # Generate URLs for scraping
    requested_lat = '[{lat}:1:{lat}]'.format(lat=lat_closest)
    requested_lon = '[{lon}:1:{lon}]'.format(lon=lon_closest)
    parameter = generate_url_params([field_id], '[0:1:23]', requested_lat, requested_lon)
    generated_URL = generate_download_links(years, database_url, database_id, parameter)
    download_manager = DownloadManager()
    download_manager.set_username_and_password(username, password)
    download_manager.download_path = field_name + '/' + loc
    download_manager.download_urls = generated_URL
    %time download_manager.start_download(NUMBER_OF_CONNECTIONS)

######### OPEN, CLEAN, MERGE, MERGE DATA AND WRITE CSVS ##########
def extract_date(data_set):
    """
    Extracts the date from the filename before merging the datasets. 
    """ 
    if 'HDF5_GLOBAL.Filename' in data_set.attrs:
        f_name = data_set.attrs['HDF5_GLOBAL.Filename']
    elif 'Filename' in data_set.attrs:
        f_name = data_set.attrs['Filename']
    else: 
        raise AttributeError('The attribute name has changed again!')
    # find a match between "." and ".nc4" that does not have "." .
    exp = r'(?<=\.)[^\.]*(?=\.nc4)'
    res = re.search(exp, f_name).group(0)
    # Extract the date. 
    y, m, d = res[0:4], res[4:6], res[6:8]
    date_str = ('%s-%s-%s' % (y, m, d))
    data_set = data_set.assign(date=date_str)
    return data_set

# Open nc4 files as dataframes, perform aggregations and save as CSV files
print('CLEANING AND MERGING DATA')
print('Predicted time: ' + str(len(years)*len(locs)*0.1) + ' minutes')
print('=====================')
for loc, lat, lon in locs:
    print('Cleaning and merging ' + field_name + ' data for ' + loc)
    dfs = []
    for file in os.listdir(field_name + '/' + loc):
        if '.nc4' in file:
            try:
                with xr.open_mfdataset(field_name + '/' + loc + '/' + file, preprocess=extract_date) as df:
                    dfs.append(df.to_dataframe())
            except:
                print('Issue with file ' + file)
    df_hourly = pd.concat(dfs)
    df_hourly['time'] = df_hourly.index.get_level_values(level=2)
    df_hourly.columns = [field_name, 'date', 'time']
    df_hourly[field_name] = df_hourly[field_name].apply(conversion_function)
    df_hourly['date'] = pd.to_datetime(df_hourly['date'])
    df_hourly.to_csv(field_name + '/' + loc + '_hourly.csv', header=[field_name, 'date', 'time'], index=False)
    df_hourly = pd.read_csv(field_name + '/' + loc + '_hourly.csv')
    df_daily = df_hourly.groupby('date').agg(aggregator)
    df_daily = df_daily.drop('time', axis=1)
    df_daily['date'] = df_daily.index
    df_daily.to_csv(field_name + '/' + loc + '_daily.csv', header=[field_name, 'date'], index=False)
    df_weekly = df_daily
    df_weekly['Week'] = pd.to_datetime(df_weekly['date']).apply(lambda x: x.isocalendar()[1])
    df_weekly['Year'] = pd.to_datetime(df_weekly['date']).apply(lambda x: x.year)
    df_weekly = df_weekly.groupby(['Year', 'Week']).agg(aggregator)
    df_weekly['Year'] = df_weekly.index.get_level_values(0)
    df_weekly['Week'] = df_weekly.index.get_level_values(1)
    df_weekly.to_csv(field_name + '/' + loc + '_weekly.csv', index=False)

print('FINISHED')

DOWNLOADING DATA FROM MERRA
Predicted time: 6 minutes


KeyboardInterrupt: 

CLEANING AND MERGING DATA
Predicted time: 0.1 minutes
Cleaning and merging tmp data for kampala
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100308.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100318.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100326.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100304.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100104.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100125.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100109.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100107.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100110.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100211.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100330.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100216.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100113.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100202.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100331.nc4
Issue with file MERRA2_300.tavg1_2d_slv_Nx.20100327.nc4
Issue wi

ValueError: No objects to concatenate